In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
MODEL = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL)
print(f"Model '{MODEL}' loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model 'sentence-transformers/all-MiniLM-L6-v2' loaded successfully.


In [ ]:
%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer

MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Load the Sentence Transformer model
model = SentenceTransformer(MODEL)

app = FastAPI()


class ChatRequest(BaseModel):
    message: str


@app.get("/")
def home():
    return {"status": "running"}


@app.get("/health")
def health():
    return {"status": "healthy"}


@app.post("/chat")
def chat(req: ChatRequest):
    # Generate embeddings for the input message
    embeddings = model.encode(req.message)

    # Return the embeddings as a list
    return {"response": embeddings.tolist()}

Writing app.py


In [ ]:
!nohup uvicorn app:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &

In [ ]:
!ps -ef | grep uvicorn

root        1623       1 85 14:10 ?        00:00:02 /usr/bin/python3 /usr/local/bin/uvicorn app:app --host 0.0.0.0 --port 8000
root        1633     547  0 14:10 ?        00:00:00 /bin/bash -c ps -ef | grep uvicorn
root        1635    1633  0 14:10 ?        00:00:00 grep uvicorn


In [ ]:
# Check the server logs
!tail -50 server.log

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4840.65it/s]
INFO:     Started server process [1623]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [ ]:
# Check the health endpoint of the FastAPI application
!curl http://127.0.0.1:8000/health

{"status":"healthy"}

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8000

2026-07-21T14:11:22Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-21T14:11:22Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-21T14:11:26Z INF +--------------------------------------------------------------------------------------------+
2026-07-21T14:11:26Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-07-21T14:11:26Z INF |  https://advisors-robinson-equipped-massachusetts.tryc